# AI Data Leak Detector

An AI-assisted security tool that scans text for potential PII, PHI, credentials, and secrets.

This version includes local redaction to reduce sensitive-data exposure before API transmission, input validation, and API error handling.

## 1. Setup

Install and import the libraries required for the scanner.

In [ ]:
!pip install openai -q

In [ ]:
from openai import OpenAI
from getpass import getpass
import os
import re

## 2. API Authentication

Securely load the OpenAI API key from a masked input and initialize the API client.

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass("API Key: ")

In [ ]:
client = OpenAI()

## 3. User Input & Validation

Collect text from the user and check that data was entered before processing.

In [ ]:
data = input("Paste internal data to scan: ").strip()

In [ ]:
if not data:
    print("Error: No data was entered. Please enter text to scan.")
else:
    print("Data received. Ready to scan.")

## 4. Local Sensitive Data Redaction

Detect and redact obvious sensitive-data patterns locally before sending text to the AI service.

In [ ]:
ssn_pattern = r"\b\d{3}-\d{2}-\d{4}\b"

In [ ]:
redacted_data = re.sub(ssn_pattern, "[REDACTED-SSN]", data)

In [ ]:
print(redacted_data)

## 5. AI Security Analysis Prompt

Build a structured security-audit prompt using the locally redacted data.

In [ ]:
prompt = f"""
You are a data security auditor.
Analyze the following internal data for sensitive information.
Check for PII, PHI, secrets, and credentials.
Return your analysis in this format:
Risk Score (1-100):
Exposed Data Types:
Sensitive Data Evidence (redacted only):
Why It Is Risky:
How To Fix It:

Do not reproduce raw passwords, credentials, secrets, PII, or PHI in your response.
Refer to sensitive values only by their data type or a redacted placeholder.

Data:
{redacted_data}
"""

## 6. Run AI Analysis

Send the sanitized prompt to the AI model and return the security analysis. API errors are handled gracefully to prevent the notebook from crashing.

In [ ]:
try:
    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )

    print(response.output_text)

except Exception as error:
    print("Error: The AI analysis could not be completed.")
    print(error)